# Caso 2 – Control de calidad en una planta de producción

**Contexto:** una empresa manufacturera produce piezas metálicas y desea analizar qué factores del proceso influyen en los defectos de fabricación.

**Variables generadas (500 registros):**

| Variable | Tipo | Unidad simulada |
|---|---|---|
| Temperatura de Producción | Numérica | °C |
| Presión de Máquina | Numérica | bar |
| Tiempo de Operación | Numérica | horas desde el último mantenimiento |
| Velocidad de Producción | Numérica | piezas/hora |
| Defectuosa | Categórica (Sí/No) | — |

Las conclusiones, el análisis de outliers y las recomendaciones a la planta están desarrolladas en el **Informe técnico** (`informe_tecnico_control_calidad.md`).

## 0. Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fijamos una semilla para que los resultados sean reproducibles
np.random.seed(42)

## 1. Generación de los 500 registros

Cada variable numérica se genera con `np.random.normal()` (distribución normal), usando valores típicos de un proceso industrial estable:

- **Temperatura de Producción** ~ Normal(media = 180 °C, desviación = 15)
- **Presión de Máquina** ~ Normal(media = 50 bar, desviación = 8)
- **Tiempo de Operación** ~ Normal(media = 8 h, desviación = 2)
- **Velocidad de Producción** ~ Normal(media = 120 piezas/h, desviación = 20)

Para que el análisis de correlación tenga sentido (y no sea pura casualidad), la probabilidad de que una pieza salga **defectuosa** se modela como una función de las cuatro variables más un término de ruido aleatorio: a mayor temperatura y mayor tiempo de operación sin mantenimiento, mayor es la probabilidad de defecto. Esto simula un escenario realista de planta.

In [ ]:
n = 500

temperatura = np.random.normal(loc=180, scale=15, size=n)
presion = np.random.normal(loc=50, scale=8, size=n)
tiempo_operacion = np.random.normal(loc=8, scale=2, size=n)
velocidad = np.random.normal(loc=120, scale=20, size=n)

# Combinación lineal (variable latente) que determina qué tan probable es el defecto
z = (
    1.8 * (temperatura - 180) / 15
    + 1.2 * (tiempo_operacion - 8) / 2
    + 0.3 * (presion - 50) / 8
    + 0.1 * (velocidad - 120) / 20
    + np.random.normal(0, 0.8, n)
)

# Función sigmoide para convertir z en una probabilidad entre 0 y 1
prob_defecto = 1 / (1 + np.exp(-z))
defectuosa = np.where(np.random.rand(n) < prob_defecto, "Sí", "No")

df = pd.DataFrame({
    "Temperatura": temperatura,
    "Presion": presion,
    "Tiempo_Operacion": tiempo_operacion,
    "Velocidad": velocidad,
    "Defectuosa": defectuosa
})

print("Dimensiones del dataset:", df.shape)
df.head()

In [ ]:
df["Defectuosa"].value_counts()

## 2. NumPy – Estadísticas descriptivas

Calculamos **media**, **mediana**, **varianza** y **desviación estándar** de cada variable numérica usando NumPy.

In [ ]:
numericas = ["Temperatura", "Presion", "Tiempo_Operacion", "Velocidad"]

estadisticas = {}
for col in numericas:
    valores = df[col].to_numpy()
    estadisticas[col] = {
        "Media": np.mean(valores),
        "Mediana": np.median(valores),
        "Varianza": np.var(valores),
        "Desviación estándar": np.std(valores),
    }

estadisticas_df = pd.DataFrame(estadisticas).T.round(2)
estadisticas_df

## 3. Matplotlib – Visualización de variables clave

### 3.1 Histograma de Temperatura

In [ ]:
plt.figure(figsize=(7,5))
plt.hist(df["Temperatura"], bins=25, color="tomato", edgecolor="black")
plt.title("Histograma - Temperatura de Producción")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Frecuencia")
plt.show()

### 3.2 Histograma de Presión

In [ ]:
plt.figure(figsize=(7,5))
plt.hist(df["Presion"], bins=25, color="steelblue", edgecolor="black")
plt.title("Histograma - Presión de Máquina")
plt.xlabel("Presión (bar)")
plt.ylabel("Frecuencia")
plt.show()

### 3.3 Gráfico de barras – Piezas defectuosas

In [ ]:
conteo_defectos = df["Defectuosa"].value_counts()

plt.figure(figsize=(6,5))
plt.bar(conteo_defectos.index, conteo_defectos.values, color=["seagreen", "crimson"])
plt.title("Piezas defectuosas vs no defectuosas")
plt.xlabel("¿Defectuosa?")
plt.ylabel("Cantidad de piezas")
plt.show()

## 4. Seaborn – Análisis de relaciones entre variables

### 4.1 Matriz de correlación

Para incluir la variable `Defectuosa` (categórica) en la matriz de correlación, primero la convertimos a numérica (`Sí` = 1, `No` = 0).

In [ ]:
df_corr = df.copy()
df_corr["Defectuosa_num"] = (df_corr["Defectuosa"] == "Sí").astype(int)

correlacion = df_corr[numericas + ["Defectuosa_num"]].corr()

plt.figure(figsize=(7,6))
sns.heatmap(correlacion, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación")
plt.show()

### 4.2 Boxplot de Temperatura

In [ ]:
plt.figure(figsize=(6,5))
sns.boxplot(x=df["Temperatura"], color="orange")
plt.title("Boxplot - Temperatura de Producción")
plt.show()

### 4.3 Pairplot

Muestra todas las relaciones cruzadas entre las variables numéricas, coloreadas según si la pieza fue defectuosa o no. Esto ayuda a ver visualmente qué variables separan mejor a las piezas defectuosas de las que no lo son.

In [ ]:
sns.pairplot(
    df, vars=numericas, hue="Defectuosa",
    palette={"Sí": "crimson", "No": "seagreen"}, diag_kind="hist"
)
plt.show()

## 5. Detección de valores atípicos (Outliers) con el método IQR

Un valor se considera atípico si cae fuera del rango `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`, donde `IQR = Q3 - Q1`.

In [ ]:
for col in numericas:
    q1 = np.percentile(df[col], 25)
    q3 = np.percentile(df[col], 75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    outliers = df[(df[col] < lim_inf) | (df[col] > lim_sup)]
    print(f"{col}: {len(outliers)} outliers | límites válidos = [{lim_inf:.2f}, {lim_sup:.2f}]")

## 6. ¿Qué variable está más asociada con los defectos?

Ordenamos la correlación de cada variable numérica con `Defectuosa_num` de mayor a menor, en valor absoluto.

In [ ]:
correlacion["Defectuosa_num"].drop("Defectuosa_num").abs().sort_values(ascending=False)

---
## Conclusiones, outliers y recomendaciones a la planta

El análisis completo de las preguntas (3.4) y el entregable final (3.5 – Informe técnico de mejora del proceso) se encuentran en **`informe_tecnico_control_calidad.md`**.